In [2]:

import os
import time
from pathlib import Path

from downloader.genome_downloader import GgetEnsemblGenomeDownloader
from genome.builder import GenomeBuilder

# Define parameters for the test
ensembl_release = 114 
assembly_id = "GRCh38"
species = "homo_sapiens"

# Track total execution time
total_start_time = time.time()

# 1. Download the files
print(f"Starting download for {species} (release {ensembl_release})...")
download_start_time = time.time()

downloader = GgetEnsemblGenomeDownloader(assembly_id=assembly_id, ensembl_release=ensembl_release, species=species)
downloaded_files = downloader.download()

download_end_time = time.time()
download_duration = download_end_time - download_start_time
print(f"Download completed in: {download_duration:.2f} seconds")

dna_path = downloaded_files['dna']
cdna_path = downloaded_files['cdna']
gtf_path = downloaded_files['annotation']

# 2. Build the Genome object from the downloaded files
print("\nBuilding Genome object...")
build_start_time = time.time()

builder = GenomeBuilder(id=assembly_id, species=species, name=f"{species} Genome (release {ensembl_release})", separate_scaffolds=True)

# Time each step separately
print("  Step 1: Loading DNA FASTA...")
dna_start_time = time.time()
builder = builder.with_dna_fasta(dna_path)
dna_end_time = time.time()
dna_duration = dna_end_time - dna_start_time
print(f"    DNA loading completed in: {dna_duration:.2f} seconds")


Starting download for homo_sapiens (release 114)...


INFO:GgetEnsemblGenomeDownloader:File 'Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz' already exists in cache. Skipping download.
INFO:GgetEnsemblGenomeDownloader:File 'Homo_sapiens.GRCh38.cdna.all.fa.gz' already exists in cache. Skipping download.
INFO:GgetEnsemblGenomeDownloader:File 'Homo_sapiens.GRCh38.114.gtf.gz' already exists in cache. Skipping download.
INFO:GenomeBuilder:Scaffold separation enabled. `build()` will return (main_genome, scaffold_genome).
INFO:GenomeBuilder:Loading DNA sequences from data/GRCh38/114/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz...
INFO:GenomeBuilder:Using existing extracted DNA FASTA file: data/GRCh38/114/Homo_sapiens.GRCh38.dna.primary_assembly.fa


Download completed in: 1.20 seconds

Building Genome object...
  Step 1: Loading DNA FASTA...


INFO:GenomeBuilder:Loaded 25 main chromosomes.
INFO:GenomeBuilder:Loaded 169 scaffold chromosomes.


    DNA loading completed in: 65.79 seconds


In [3]:

print("  Step 2: Loading cDNA FASTA...")
cdna_start_time = time.time()
builder = builder.with_cdna_fasta(cdna_path)
cdna_end_time = time.time()
cdna_duration = cdna_end_time - cdna_start_time
print(f"    cDNA loading completed in: {cdna_duration:.2f} seconds")


INFO:GenomeBuilder:Loading cDNA sequences from data/GRCh38/114/Homo_sapiens.GRCh38.cdna.all.fa.gz...
INFO:GenomeBuilder:Reading gzipped cDNA FASTA file: data/GRCh38/114/Homo_sapiens.GRCh38.cdna.all.fa.gz


  Step 2: Loading cDNA FASTA...


INFO:GenomeBuilder:Loaded 207175 cDNA sequences.


    cDNA loading completed in: 4.80 seconds


In [4]:

print("  Step 3: Loading GTF annotation...")
gtf_start_time = time.time()
builder = builder.with_gtf_file(gtf_path)
gtf_end_time = time.time()
gtf_duration = gtf_end_time - gtf_start_time
print(f"    GTF loading completed in: {gtf_duration:.2f} seconds")


INFO:GenomeBuilder:Processing annotations from data/GRCh38/114/Homo_sapiens.GRCh38.114.gtf.gz...
INFO:GenomeBuilder:Loading existing gffutils database: data/GRCh38/114/Homo_sapiens.GRCh38.114.gtf.db


  Step 3: Loading GTF annotation...


INFO:root:GTF database created at: data/GRCh38/114/Homo_sapiens.GRCh38.114.gtf.db
INFO:GenomeBuilder:Created genes in 7.29 seconds
INFO:GenomeBuilder:Created transcripts in 27.85 seconds
INFO:GenomeBuilder:Created exons in 103.13 seconds
INFO:GenomeBuilder:Successfully parsed and linked 78894 genes, 387954 transcripts.


    GTF loading completed in: 138.49 seconds


In [5]:

print("  Step 4: Building final genome object...")
final_build_start_time = time.time()
genome, scaffold_genome = builder.build()
final_build_end_time = time.time()
final_build_duration = final_build_end_time - final_build_start_time
print(f"    Final build completed in: {final_build_duration:.2f} seconds")

INFO:GenomeBuilder:Indexing genome for fast lookups...


  Step 4: Building final genome object...


INFO:GenomeBuilder:Indexing scaffold genome for fast lookups...
INFO:GenomeBuilder:Genome construction complete.
INFO:GenomeBuilder:Offloading builder memory...
INFO:GenomeBuilder:Memory offload complete.


    Final build completed in: 5.73 seconds


In [6]:
# Get all protein coding genes
protein_coding_genes = [gene for gene in genome.genes if gene.gene_biotype == "protein_coding"]
print(f"Number of protein coding genes: {len(protein_coding_genes)}")


Number of protein coding genes: 20096


In [11]:
# Get all protein coding genes
scaffold_genome.genes[0].transcripts[0].sequence


''